# 03 · Neural Bayes estimation of extremal dependence

> **Source.** The code in this notebook comes from the CASE 2025 short course *AI and extremes* by **Likun Zhang, Jordan Richards and Reetam Majumder** ([Jbrich95/CASE2025_shortcourse](https://github.com/Jbrich95/CASE2025_shortcourse)). It is converted from the original `.Rmd` to a Google Colab notebook, with explanatory notes added. Changes to the original code are marked **[Adapted]** and explained where they occur.

**How to run on Google Colab.** Open this notebook in Colab, choose *Runtime → Change runtime type → R* (CPU is enough), then *Runtime → Run all*. The setup cell installs the R packages and downloads the data from the course repository. The first time Keras is used, R may spend a few minutes setting up Python/TensorFlow.

**What this notebook covers.** For many spatial-extremes models the likelihood is intractable or very expensive, but *simulating* from the model is easy. A **neural Bayes estimator** exploits this:

1. Draw parameters $\theta_k$ from a prior.
2. Simulate a dataset $\mathbf Z_k$ from the model with parameter $\theta_k$.
3. Train a network $\hat\theta(\cdot)$ to map $\mathbf Z_k \mapsto \theta_k$ by minimising mean-squared error.

Under squared-error loss, the Bayes estimator is the **posterior mean** $E[\theta \mid \mathbf Z]$, so the trained network approximates it. Once trained, the estimator is **amortised**: it gives an estimate for any new dataset in milliseconds, with no refitting.

Here: the dependence parameter $\alpha$ of the bivariate **Hüsler–Reiss** extreme-value distribution, applied to annual temperature maxima at pairs of sites. Background: Sainsbury-Dale, Zammit-Mangion & Huser (2024, *The American Statistician*); Richards, Sainsbury-Dale, Zammit-Mangion & Huser (2024, *JMLR*).

In [ ]:
# [Adapted] Setup that works on Google Colab's R runtime and on a local machine.
options(timeout = 1200)                       # large data file on slow connections
on_colab <- dir.exists("/content")
if (on_colab) install.packages(c("keras3", "evd"))
# Local machines: use a Python virtualenv called "r-keras" if one exists
if (!on_colab && reticulate::virtualenv_exists("r-keras")) {
  reticulate::use_virtualenv("r-keras", required = TRUE)
}
library(keras3)

# Course data, downloaded from the original repository if not already present
base_url <- "https://github.com/Jbrich95/CASE2025_shortcourse/raw/main/"
for (f in c("weather.RDS")) {
  if (!file.exists(f)) download.file(paste0(base_url, f), f, mode = "wb")
}
dir.create("checkpoints", showWarnings = FALSE)   # [Adapted] folder used by the checkpoint callbacks

# If the first Keras call fails with a Python/TensorFlow error, run
#   keras3::install_keras()
# then restart the session (Runtime -> Restart session) and run this cell again.

In [ ]:
set.seed(1)
tensorflow::set_random_seed(1)

## 1 · Simulate training data

Prior $\alpha \sim \text{Gamma}(1, 1)$. For each draw, simulate $n = 64$ bivariate Hüsler–Reiss observations on standard Gumbel margins. The value 64 matches the number of years of real data used later.

> **[Adapted] note.** Simulating 50,000 datasets takes a minute or two.

In [ ]:
prior_alpha = function(K) {
  rgamma(K, 1, 1)
}

K <- 50000
n <- 64
alpha.train <- prior_alpha(K)
Z.train <- t(apply(as.matrix(alpha.train), 1, function(x)
  c(evd::rbvevd(n, dep = x, model = "hr"))))
dim(Z.train)

Each simulated dataset is flattened into a vector of length $2n = 128$: all 64 values for site 1, then all 64 for site 2. An MLP maps it to $\hat\alpha$, with an exponential output so that $\hat\alpha > 0$.

In [ ]:
model <- keras_model_sequential() %>%
  layer_dense(units = 64, activation = 'relu') %>%
  layer_dense(units = 64, activation = 'relu') %>%
  layer_dense(units = 1, activation = 'exponential')

model %>% compile(optimizer = "adam", loss = "mean_squared_error")

early.stopping <- callback_early_stopping(monitor = "val_loss", patience = 10)
history <- model %>% fit(
  x = Z.train, y = as.matrix(alpha.train),
  callbacks = list(early.stopping),
  epochs = 100, verbose = 0,
  validation_split = 0.2, shuffle = T
)
plot(history)
summary(model)

## 2 · Check the estimator on fresh simulations

This evaluation is clean: the 1,000 test datasets are **new** simulations that the network has never seen. This is the "controlled simulation study where tail properties are known" idea from the E5 project's validation plan.

In [ ]:
K.test <- 1000
alpha.test <- prior_alpha(K.test)
Z.test <- t(apply(as.matrix(alpha.test), 1, function(x)
  c(evd::rbvevd(n, dep = x, model = "hr"))))

predictions <- model %>% predict(Z.test)
plot(alpha.test, predictions)
abline(a = 0, b = 1)

## 3 · Apply to real data: annual temperature maxima

Extract annual maxima at two nearby sites.

In [ ]:
data <- readRDS("weather.RDS")
data = data[data$loc == 1 | data$loc == 2, ]
years = unique(format(as.Date(data$time), "%Y"))

int <- 1
tmax <- matrix(0, nrow = length(years), ncol = 2)
for (t in years) {
  inds = which(format(as.Date(data$time), "%Y") == t & data$loc == 1)
  tmax[int, 1] = max(data$tmax[inds])
  inds = which(format(as.Date(data$time), "%Y") == t & data$loc == 2)
  tmax[int, 2] = max(data$tmax[inds])
  int = int + 1
}

# [Adapted] The estimator was trained for exactly n = 64 replicates
stopifnot(nrow(tmax) == n)
plot(tmax)

Transform to standard Gumbel margins to match the training data. Each margin is mapped to uniform with its **empirical ranks**, then to Gumbel with the GEV(0, 1, 0) quantile function.

> **Interpretation note.** Rank-based margins ignore uncertainty in the marginal fit, which is standard in dependence modelling but worth knowing.

In [ ]:
unif = function(x) rank(x) / (length(x) + 1)
tmax.Gumbel = evd::qgev(apply(tmax, 2, unif))
plot(tmax.Gumbel)

alpha.est <- model %>% predict(t(as.matrix(c(tmax.Gumbel))))
alpha.est

Check the fit visually: overlay density contours of 100,000 draws from the fitted model.

In [ ]:
plot(tmax.Gumbel)
sims <- evd::rbvevd(100000, alpha.est, model = "hr")
biv_kde <- MASS::kde2d(sims[, 1], sims[, 2])
contour(biv_kde, add = T, col = "red")

## 4 · Re-use the same estimator (amortisation)

Sites 1 and 25 are further apart. Their maxima look much less dependent in the tail, and the same trained network gives a new estimate instantly.

In [ ]:
data <- readRDS("weather.RDS")
data = data[data$loc == 1 | data$loc == 25, ]
years = unique(format(as.Date(data$time), "%Y"))

int <- 1
tmax <- matrix(0, nrow = length(years), ncol = 2)
for (t in years) {
  inds = which(format(as.Date(data$time), "%Y") == t & data$loc == 1)
  tmax[int, 1] = max(data$tmax[inds])
  inds = which(format(as.Date(data$time), "%Y") == t & data$loc == 25)
  tmax[int, 2] = max(data$tmax[inds])
  int = int + 1
}
stopifnot(nrow(tmax) == n)

tmax.Gumbel = evd::qgev(apply(tmax, 2, unif))
alpha.est <- model %>% predict(t(as.matrix(c(tmax.Gumbel))))
alpha.est

plot(tmax.Gumbel)
sims <- evd::rbvevd(100000, alpha.est, model = "hr")
biv_kde <- MASS::kde2d(sims[, 1], sims[, 2])
contour(biv_kde, add = T, col = "red")

## Summary and limitations

* A neural Bayes estimator learns the posterior mean from simulations alone and is amortised once trained.
* **Fixed sample size.** This network only accepts exactly $n = 64$ replicates.
* **Not permutation-invariant.** The data enter as one flat vector, so re-ordering the years changes the input even though the information is the same. DeepSets-style architectures, as in the `NeuralEstimators` package, fix both issues by pooling a summary over replicates.
* **Point estimate only.** Uncertainty needs extra work, for example quantile-loss estimators or a parametric bootstrap that re-uses the amortised network.
* **The estimate depends on the prior.** It approximates the posterior mean under the Gamma(1, 1) prior, which matters most when there is little data.